# Quantization Aware Training

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader


from utils.preprocessing import prepare_dataloaders, load_mobilenetv2_model, load_model
from utils.metrics import compute_top_k_accuracy, benchmark_model, print_model_size
from utils.post_training import get_custom_qconfig

In [ ]:
# Specify random seed for repeatable results
_ = torch.manual_seed(191009)

### Hyperparameters

In [ ]:
# Paths for the dataset and model weights
data_dir = '../../../../EdgeComputingGroup/model-compression/data/skin-lesions/download'  # Update with your dataset path
model_weights_path = '../../mobilenet_v2_best_model.pth'  # Update with your saved model weights

batch_size = 32

# Device configuration - quantization is often done on CPU.
device = torch.device("cpu")

### Helper Functions

In [ ]:
def load_quantized_model(model_path: str, device: torch.device = torch.device("cpu")) -> torch.jit.ScriptModule:
    """
    Loads a quantized TorchScript model from the specified file.

    Parameters:
        model_path (str): Path to the saved TorchScript model (.pt file).
        device (torch.device): The device on which to load the model. Defaults to CPU.
    
    Returns:
        torch.jit.ScriptModule: The loaded quantized model.
    """
    # Load the TorchScript model from disk, mapping it to the specified device.
    model = torch.jit.load(model_path, map_location=device)
    # Set the model to evaluation mode (important for inference)
    model.eval()
    return model


def evaluate_model(model: torch.nn.Module, data_loader: DataLoader, criterion: nn.Module) -> None:
    """
    Evaluates the model on the evaluation data and prints accuracy.
    
    Parameters:
        model (nn.Module): The model to evaluate.
        data_loader (DataLoader): DataLoader for the evaluation dataset.
        criterion (nn.Module): Loss function.
    """
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_samples = 0
    with torch.no_grad():
        for images, targets in data_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * images.size(0)
            acc1 = compute_top_k_accuracy(outputs, targets, topk=(1,))[0]
            total_top1 += acc1 * images.size(0) / 100.0  # converting percent back to count
            total_samples += images.size(0)
    
    avg_loss = total_loss / total_samples
    avg_acc = total_top1 / total_samples * 100.0
    print(f"Evaluation Loss: {avg_loss:.4f}, Top-1 Accuracy: {avg_acc:.2f}%")


### Quantization Aware Training

In [ ]:

def train_qat_model(model: nn.Module, train_loader: DataLoader, device: torch.device, epochs: int = 10, lr: float = 1e-4) -> nn.Module:
    """
    Trains the QAT model using a standard training loop with cross entropy loss

    Parameters:
        model (nn.Module): The QAT-prepared model.
        train_loader (DataLoader): DataLoader for the training dataset.
        device (torch.device): Device to perform training on.
        epochs (int): Number of training epochs.
        lr (float): Learning rate.
    
    Returns:
        nn.Module: The trained QAT model.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()  # Ensure model is in training mode
        running_loss = 0.0

        # Create a progress bar for the current epoch
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        
        for images, targets in progress_bar:
            images, targets = images.to(device), targets.to(device)
            
            optimizer.zero_grad()               # Reset gradients
            outputs = model(images)             # Forward pass
            loss = criterion(outputs, targets)  # Compute loss
            loss.backward()                     # Backward pass
            optimizer.step()                    # Update weights
            
            # Update running loss
            running_loss += loss.item() * images.size(0)
            
            # Update progress bar with current batch loss
            progress_bar.set_postfix(loss=loss.item())
        
        # Calculate and print epoch loss
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
    
    return model

### Main

In [ ]:
def qat_main():
    # Prepare data loaders
    train_loader, eval_loader = prepare_dataloaders(data_dir, 32)

    num_class = len(train_loader.dataset.classes)

    # Load the pre-trained (fine-tuned) MobileNetV2 model
    qat_model = load_mobilenetv2_model(model_weights_path, num_class, device, "qat")
    qat_model.train()
    qat_model.fuse_model(is_qat=True)
    optimizer = torch.optim.SGD(qat_model.parameters(), lr = 0.0001)

    # Use the custom qconfig in your FX quantization flow
    # custom_qconfig = get_custom_qconfig()
    # qat_model.qconfig = torch.ao.quantization.QConfigMapping().set_global(custom_qconfig)
    qat_model.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')

    torch.ao.quantization.prepare_qat(qat_model, inplace=True)
    # print('Inverted Residual Block: After preparation for QAT, note fake-quantization modules \n',qat_model.features[1].conv)

    model = train_qat_model(qat_model, train_loader, device, epochs=1, lr=1e-4)


In [ ]:
# model = torch.ao.quantization.convert(model.eval(), inplace=False)
# print_model_size(model)

# torch.jit.save(torch.jit.script(model), "../../mobilenetv2_quantized.pt")

# model_test = load_quantized_model("../../mobilenetv2_quantized.pt")